In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import multiprocessing
import os
import pickle

if os.environ.get("SLURM_CPUS_PER_TASK"):
    cores = int(os.environ.get("SLURM_CPUS_PER_TASK", 1))
else:
    cores = multiprocessing.cpu_count()
print(f"Number of cores: {cores}")

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count={}".format(cores)

import jax
import jax.numpy as jnp
import numpy as np
from jax import jit
from jaxoplanet import orbits
from jaxoplanet.light_curves import LimbDarkLightCurve
from jaxopt import ScipyMinimize
from tensorflow_probability.substrates.jax.bijectors import Log
from tensorflow_probability.substrates.jax.distributions import Normal

from gallifrey.gps import get_trainables, kernel_summary
from gallifrey.inference import log_likelihood_function
from gallifrey.inference.mcmc import (
    nuts_warmup,
    run_mcmc,
)
from gallifrey.transit import example_lightcurve
from gallifrey.util import dict_to_jnp

rng_key = jax.random.PRNGKey(42)

## LOAD DATA AND MODEL

In [ ]:
example = example_lightcurve()

model_name = example["name"] + "_rbf_kernel"
with open(f"../data/processed/toy_data/gp_models/{model_name}", "rb") as file:
    model = pickle.load(file)

summary = kernel_summary(model, to_latex=False)

## CREATE TRANSIT MODEL

In [ ]:
@jit
def lc_model(t, params):
    orbit = orbits.keplerian.Body(
        period=15,
        radius=params["radius"],
        inclination=jnp.deg2rad(89),
        time_transit=0,
    )

    lc = LimbDarkLightCurve([params["u1"], params["u2"]]).light_curve(orbit, t=t)
    return lc

## FIT LC

In [ ]:
initial_position = {
    "gp_parameter": get_trainables(
        model, unconstrain=True
    ),  # gp is fitted in unconstrained space, so set initial value to that
    "lc_parameter": {"radius": 0.2, "u1": 0.2, "u2": 0.5},
}

param_priors = {
    "gp_parameter": Normal(loc=initial_position["gp_parameter"], scale=1),
    "lc_parameter": Normal(
        loc=dict_to_jnp(initial_position["lc_parameter"]),
        scale=[0.1, 0.3, 0.2],
    ),
}

In [ ]:
log_likelihood = log_likelihood_function(
    model,
    lc_model,
    example["t_sample"],
    example["lc_sample"],
    example["transit_mask_sample"],
    fix_gp=False,
    compile=True,
)


@jit
def log_priors(params):
    gp_log_priors = param_priors["gp_parameter"].log_prob(params["gp_parameter"])
    lc_log_priors = param_priors["lc_parameter"].log_prob(
        dict_to_jnp(params["lc_parameter"])
    )
    return jnp.sum(gp_log_priors) + jnp.sum(lc_log_priors)


@jit
def log_probability(params):
    return log_likelihood(params) + log_priors(params)


neg_log_probability = jit(lambda params: -log_probability(params))

In [ ]:
lbfgsb = ScipyMinimize(fun=neg_log_probability, method="l-bfgs-b")
lbfgsb_sol = lbfgsb.run(initial_position)
print("Best fit parameter: ", lbfgsb_sol.params["lc_parameter"])

## RUN MCMC

In [ ]:
# Adapted from BlackJax's introduction notebook.
num_adapt = 100
num_samples = 500
num_chains = cores

rng_key, warmup_key, sample_key = jax.random.split(rng_key, 3)

state, parameters = nuts_warmup(
    warmup_key,
    log_probability,
    initial_position,
    num_steps=num_adapt,
)

initial_positions = {
    "gp_parameter": jnp.tile(state.position["gp_parameter"], (num_chains, 1)),
    "lc_parameter": {
        param: jnp.tile(state.position["lc_parameter"][param], (num_chains, 1)).reshape(
            -1
        )
        for param in initial_position["lc_parameter"].keys()
    },
}

final_state, state_history, info_history = run_mcmc(
    sample_key,
    log_probability,
    parameters,
    initial_positions,
    num_steps=num_samples,
)

In [ ]:
gp_parameter = dict_to_jnp(state_history.position["gp_parameter"])
lc_parameter = dict_to_jnp(state_history.position["lc_parameter"])

np.save(
    f"../data/processed/toy_data/mcmc_chains/{model_name}_parameter.npy",
    lc_parameter.reshape(-1, lc_parameter.shape[-1]),
)
np.save(
    f"../data/processed/toy_data/mcmc_chains/{model_name}_parameter_gp.npy",
    gp_parameter.reshape(-1, gp_parameter.shape[-1]),
)